# Hands-on CURVE

Acknowledgements: [Stefan Oehmcke](https://stefoe.github.io/) for the code used in this notebook.

This notebook will guide you through a data preprocessing pipeline implementing the CURVE framework you heard about in the lecture.

- CURVE: Clean – Unite – Record – Validate – Export
- Dataset: Our World in Data (OWID) CO2 + Energy (open, no API key)
- Question: Is a higher low-carbon electricity share associated with lower CO2 per capita (2000–2024)?

## Initial configuration

Import statements are in the first code cell, so the order in which later cells are executed only depends on what they are doing, not what packages they need (because all the packages are already loaded).

Also placing variables that configure high-level behaviour at the top of your notebooks makes it easier to repurpose a notebook.

In [ ]:
import pandas as pd
import numpy as np
from datetime import date
import platform
import matplotlib.pyplot as plt

YEARS = (2000, 2024)
MISSING_POLICY = "drop"         # options: "drop" | "interpolate"
INTERP_LIMIT = 2                # max consecutive years to fill when interpolating
EXPORT_BASENAME = f"owid_co2_energy_ready_{date.today().isoformat()}"

CO2_URL    = "https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv"
ENERGY_URL = "https://raw.githubusercontent.com/owid/energy-data/master/owid-energy-data.csv"

## Load data

:::{important} Tasks
1. Use [pandas.read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) to load each of the datasets.
:::

In [ ]:
co2_raw = None  # TODO
co2_raw

In [ ]:
energy_raw = None  # TODO
energy_raw

# C = Clean

## Select only necessary columns

We don't need all of the many columns that are available.
This selects only a necessary subset of columns.

In [ ]:
co2_keep = ["iso_code", "country", "year", "co2_per_capita"]
co2 = co2_raw[co2_keep].copy()
co2

In [ ]:
# Prefer low_carbon_electricity_share; fall back to renewables_share_elec if needed
cands = [c for c in energy_raw.columns if "low_carbon_electricity_share" in c] \
        or [c for c in energy_raw.columns if "renewables_share_elec" in c]
share_col = cands[0]

energy_keep = ["iso_code", "country", "year", share_col, "electricity_demand", "population", "gdp"]
energy = energy_raw[energy_keep].copy()
energy = energy.rename(columns={share_col: "low_carbon_elec_share"})  # whatever share column we end up using, give it a fixed name
energy_keep = ["iso_code", "country", "year", "low_carbon_elec_share", "electricity_demand", "population", "gdp"]
energy

:::{important} Tasks
2. Looking at the printouts of the dataframes, does the data meet the tidy criteria from the lecture?
:::

:::{tip} Your answer
:class:dropdown
2.
:::

## Handle missing data

First have a look at where data is missing.
We do this by printing which columns include missing values.

In [ ]:
for col in co2_keep:
    no_na = co2[col].notna().all()
    print(f"{col}: {'complete' if no_na else 'some missing'}")

In [ ]:
for col in energy_keep:
    no_na = energy[col].notna().all()
    print(f"{col}: {'complete' if no_na else 'some missing'}")

:::{important} Tasks
3. Implement the case in which missing data should be dropped.
:::

In [ ]:
def handle_missing(data, req_cols):
    # Missing data policy: DROP vs INTERPOLATE
    if MISSING_POLICY == "drop":
        # TODO
        pass
    elif MISSING_POLICY == "interpolate":
        # Interpolate within each country over time (linear), up to INTERP_LIMIT consecutive years
        data = data.sort_values(["iso_code", "year"])

        def _interp_country(g):
            g = g.copy()
            for c in req_cols:
                g[c] = g[c].interpolate(method="linear", limit=INTERP_LIMIT, limit_direction="both")
                # optional: clamp to valid share range after interpolation
                if c == "low_carbon_elec_share":
                    g["low_carbon_elec_share"] = g["low_carbon_elec_share"].clip(lower=0, upper=100)
            return g

        na_before = data[req_cols].isna().sum().sum()
        data = data.groupby("iso_code", group_keys=False).apply(_interp_country)
        na_after = data[req_cols].isna().sum().sum()
        print(f"[Missing policy: INTERPOLATE] NA cells before: {na_before}  after: {na_after}")

        # If some NAs remain (long gaps), drop them so validation is simple for students.
        if na_after > 0:
            rows_before = len(data)
            data = data.dropna(subset=req_cols)
            print(f"Remaining NA rows removed after interpolation: {rows_before - len(data)}")
        
        print("Comment: OK for short, smooth gaps; document the assumption. Avoid long-gap fills or extrapolation.")
    else:
        raise ValueError("MISSING_POLICY must be 'drop' or 'interpolate'.")

    return data

In [ ]:
# cannot interpolate -> drop
co2 = co2.dropna(subset=["iso_code"])
energy = energy.dropna(subset=["iso_code"])

# interpolate if requested
co2 = handle_missing(co2, ["co2_per_capita"])
energy = handle_missing(energy, ["low_carbon_elec_share"])

:::{important} Tasks
We only need data for the specified time range, that was configured in the very first code cell of this notebook.

4. Select rows in both `co2` and `energy` in the closed interval defined by `YEARS`.
:::

In [ ]:
co2 = co2.query("")  # TODO: add the query
energy = energy.query("")  # TODO: add the query

In [ ]:
print(f"#co2 records: {len(co2)}")  # should be 5325
print(f"#energy records: {len(energy)}")  # should be 5243

## Inspecting the data

In [ ]:
co2.describe()

In [ ]:
energy.describe()

:::{important} Tasks
5. Does this indicate that the correct range of years was selected?
6. Is the data roughly equally distributed across the years? (Based on percentiles)
:::

:::{tip} Your answer
:class:dropdown
5.
6.
:::

# U = UNITE

:::{important} Tasks
We want to combine the `co2` and `energy` dataframes by finding matching rows (based on the `iso_code` **and** `year` columns) and merging each match into a new row, creating a new dataframe.

7. Implement this operation using the [pandas.DataFrame.merge](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html) method and assign it to a new variable called `df`.

Read into the documentation of the `how` parameter to find out which kind of merge operation you need here.
:::

In [ ]:
# TODO
df = None

In [ ]:
print(f"#merged records: {len(df)}")  # should be 4976

# R = RECORD

:::{important} Tasks
8. What are the data sources?
9. What major decisions on how to handle the data have been taken?
    1. Was any data removed?
    2. Were data types changed?
    3. Were columns/rows/cells merged or split?
    4. Were identifiers of columns/rows changed?

A systematic and extended version of this was published in the [ODE protocol](https://www.sciencedirect.com/science/article/pii/S1364815226000599) (Table 1).
:::

:::{tip} Your answer
:class:dropdown
8.
9.
    1.
    2.
    3.
    4.
:::

# V = VALIDATE

## Check essential properties

:::{important} Tasks
10. Implement an assertion, that checks that there are no repeated ISO codes or years.
11. Implement an assertion, that checks that all values in `low_carbon_elec_share` are within the interval [0,100].
:::

In [ ]:
# 1) Keys unique per (iso_code, year)
# TODO

# 2) Required fields not NA
assert ~df[["co2_per_capita", "low_carbon_elec_share"]].isna().any().any(), "Missing values remain in required columns."

# 3) Value ranges
assert (df["co2_per_capita"] >= 0).all(), "Negative CO2 per capita values detected."
# TODO

# 4) Optional plausibility cap (very high extremes rarely occur)
assert df.loc[(df["co2_per_capita"] > 80) | (df["low_carbon_elec_share"] > 100)].empty, "Implausible extremes present."

print("Validation passed")

## Quick sanity outputs

In [ ]:
n_countries = df["iso_code"].nunique()
n_rows = len(df)
print(f"Countries: {n_countries} | Rows: {n_rows} | Years: [{df['year'].min()}–{df['year'].max()}]")

# Simple correlation
corr = df[["co2_per_capita", "low_carbon_elec_share"]].corr().iloc[0,1]
print(f"Pearson corr(co2_per_capita, low_carbon_elec_share) = {corr:.3f}")

# Tiny scatter for a feel (random 3 countries)
sample_c = df["iso_code"].drop_duplicates().sample(n=min(3, n_countries), random_state=0).tolist()
sub = df[df["iso_code"].isin(sample_c)]
plt.figure(figsize=(6,4))
for c in sample_c:
    cc = sub[sub["iso_code"]==c]
    plt.scatter(cc["low_carbon_elec_share"], cc["co2_per_capita"], label=c, s=18, alpha=0.8)
plt.xlabel("Low-carbon electricity share (%)")
plt.ylabel("CO2 per capita (t/person)")
plt.title("Sample countries across 2000–2024")
plt.legend(title="ISO")
plt.tight_layout()
plt.show()

# E = EXPORT

:::{important} Tasks
12. Save the contents of `df` in CSV format without the index using [pandas.DataFrame.to_csv](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html).
:::

## Reproduction notes

:::{important} Tasks
13. Print version information for your installation of Python as well as used packages.
14. Print configuration values.
15. Print column names, that you saved in the previous step.
:::

:::{important} Tasks
Check out the [pyproject.toml](https://github.com/bckrlab/data-science-course/blob/main/pyproject.toml) file in the underlying git repository.

16. How much of this printout is already defined there?
17. Research/Discuss: Why is it better to have a file like pyproject.toml compared to the printouts?
:::

:::{tip} Your answer
:class:dropdown
16.
17.
:::